# 워밍업 — 같은 모델, 세 개의 운명 🎢

웹 챕터에서 본 두 실험을 이번엔 **직접 코드로** 돌려 봅니다.

> **노트북 사용법**: 각 셀(회색 상자)을 클릭하고 `Shift + Enter`를 누르세요. 위에서 아래로 반복하면 끝.
> 코드를 다 이해하지 못해도 괜찮아요. 실행해서 결과를 눈으로 보는 게 목적입니다.

**이 노트북의 목표**: 8부 전체의 출발 질문을 코드로 확인한다.
**"수학(기울기)이 완벽히 같아도, 다이얼 하나가 학습의 운명을 가른다."**

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 오른쪽 아래 버튼을 눌러 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('sample')

## 1. 무대: 2부 9장의 경사하강, 코드 세 줄

손실 골짜기 L(w) = (w−3)², 출발점 w = −5. 갱신 공식은 그 한 줄입니다:
**새 위치 = 지금 위치 − 보폭 × 기울기**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

TARGET, START = 3.0, -5.0
loss = lambda w: (w - TARGET) ** 2
grad = lambda w: 2 * (w - TARGET)          # 미분 (2부 4장의 거듭제곱 공식)

def train(lr, steps=20):
    """경사하강 — 보폭(lr)만 인자로 받는다. 난수 없음(완전 결정적)."""
    w, hist = START, [START]
    for _ in range(steps):
        w = w - lr * grad(w)               # 새 위치 = 지금 위치 − 보폭 × 기울기
        hist.append(w)
    return np.array(hist)

print('준비 완료: 같은 train() 함수를 보폭만 바꿔 세 번 부릅니다')

## 2. 실험 ①: 세 개의 운명

In [ ]:
runners = {'🐢 소심이 (lr 0.02)': 0.02, '🏃 적당이 (lr 0.3)': 0.3, '💥 무모이 (lr 1.05)': 1.05}

plt.figure(figsize=(6.5, 3.8))
for (name, lr), color in zip(runners.items(), ['tab:blue', 'tab:green', 'tab:red']):
    hist = train(lr)
    L = loss(hist)
    plt.plot(np.minimum(L, 300), label=name, color=color)   # 폭발은 300에서 잘라 표시
    print(f'{name}: 20스텝 후 손실 {L[-1]:.4f}')
plt.xlabel('스텝'); plt.ylabel('손실 (300에서 절단)'); plt.legend(); plt.grid(alpha=0.3)
plt.title('같은 수학, 다른 보폭: 세 개의 운명'); plt.tight_layout(); plt.show()

slow, good, boom = (loss(train(lr))[-1] for lr in runners.values())
assert slow > 5, '소심이: 방향은 맞지만 20스텝으로는 한참 남았다 (기어가기)'
assert good < 0.01, '적당이: 몇 스텝 만에 골짜기 바닥 도착'
assert boom > loss(START), '무모이: 출발점보다 높다 — 폭발(발산)'
print('\n✔ 확인: 기울기 계산은 셋 다 같았다. 운명을 가른 건 보폭(학습률) 하나')

## 3. 실험 ② · 경계 탐색: 폭발은 정확히 어디서 시작되나

보폭을 0.1부터 1.15까지 훑으며 20스텝 후 손실을 봅니다. **절벽**이 보일 거예요.

In [ ]:
print(' 보폭(lr)   20스텝 후 손실     운명')
for lr in [0.1, 0.3, 0.6, 0.9, 0.99, 1.0, 1.01, 1.1, 1.15]:
    L = loss(train(lr))[-1]
    fate = ('💥 폭발' if L > loss(START) + 1e-9 else
            '♾️ 무한 진동' if abs(L - loss(START)) < 1e-9 else
            '✅ 도착' if L < 0.01 else '🐢 가는 중')
    print(f'   {lr:5.2f}   {L:15.6f}   {fate}')

# 경계의 수학적 확인 — 이 골짜기에서는 정확히 lr = 1.0
assert loss(train(0.99))[-1] < loss(START), 'lr < 1: 언젠가 도착하는 쪽'
assert abs(loss(train(1.0))[-1] - loss(START)) < 1e-9, 'lr = 1.0: 같은 높이를 영원히 왕복'
assert loss(train(1.01))[-1] > loss(START), 'lr > 1: 폭발'
print('\n✔ 확인: 도착↔폭발의 경계는 정확히 1.00. 운이 아니라 계산 가능한 경계 (왜 1인지는 2장)')

**직접 해보기**: 위 `train(lr, steps=20)` 의 스텝 수를 200으로 늘리면 소심이(0.02)도 도착할까요?
바꿔서 실행해 보세요. "작으면 안전하지만 시간이 돈"이라는 트레이드오프가 몸에 들어옵니다.

## 4. 관찰 정리 + 🤖 AI 연결

- **학습률**(learning rate, lr) = 경사하강 한 걸음의 보폭. 설정 파일의 `lr` 이 이것.
- 너무 작으면 🐢 하세월, 적당하면 ✅ 몇 스텝, 너무 크면 💥 골짜기를 건너뛰며 폭발(**발산**).
  그리고 그 사이엔 **날카로운 경계**(이 골짜기에선 정확히 1.00)가 있다. 결정적이라 다시 돌려도 같은 운명.
- 학습률처럼 **사람이 훈련 전에 정하는 다이얼**의 총칭이 **하이퍼파라미터**. 8부가 다루는 과목입니다.
- 다시 쓰는 수학: **2부 9장 경사하강**. 2부가 기울기를 맡았다면, 8부는 보폭과 그 친구들.

**다음 챕터**: 1장(안개 낀 산과 여행자). 오늘의 1차원 골짜기를 수십억 차원의
**손실 지형**(loss landscape)으로 확장한 전체 지도를 폅니다.